In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [3]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
338,After 21 movies and three years of working in ...,positive
954,I first saw this in the theater in 1969 when I...,positive
769,Can I just start by saying I'm a fan of bad mo...,negative
617,Sitting in a big wing chair with a huge book i...,negative
128,"I may not have the longest of attention-spans,...",negative


In [4]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [5]:
df = normalize_text(df)
df.head()

,review,sentiment
338,movie three year working hollywood bette davis...,positive
954,first saw theater immediately fell love it sad...,positive
769,start saying fan bad movie really bad movie st...,negative
617,sitting big wing chair huge book lap one bela ...,negative
128,may longest attention span second movie refuse...,negative


In [6]:
df['sentiment'].value_counts()

sentiment
negative    251
positive    249
Name: count, dtype: int64

In [7]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
338,movie three year working hollywood bette davis...,1
954,first saw theater immediately fell love it sad...,1
769,start saying fan bad movie really bad movie st...,0
617,sitting big wing chair huge book lap one bela ...,0
128,may longest attention span second movie refuse...,0


In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:
vectorizer = CountVectorizer(max_features=150)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
import dagshub
import dotenv
dotenv.load_dotenv()
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
dagshub.init(repo_owner=os.getenv("MLFLOW_TRACKING_USERNAME"), repo_name="MLOPS-PROJ-CI-CD", mlflow=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-07-31 17:55:28,167 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/satyagudu1146/MLOPS-PROJ-CI-CD "HTTP/1.1 200 OK"


Initialized MLflow to track repo "satyagudu1146/MLOPS-PROJ-CI-CD"

2026-07-31 17:55:28,174 - INFO - Initialized MLflow to track repo "satyagudu1146/MLOPS-PROJ-CI-CD"


Repository satyagudu1146/MLOPS-PROJ-CI-CD initialized!

2026-07-31 17:55:28,178 - INFO - Repository satyagudu1146/MLOPS-PROJ-CI-CD initialized!


<Experiment: artifact_location='mlflow-artifacts:/c01a4b9d46d44f35821debda17e70ec4', creation_time=1785500210662, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1785500210662, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [20]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 150)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-31 17:55:29,831 - INFO - Starting MLflow run...
2026-07-31 17:55:30,411 - INFO - Logging preprocessing parameters...
2026-07-31 17:55:31,783 - INFO - Initializing Logistic Regression model...
2026-07-31 17:55:31,784 - INFO - Fitting the model...
2026-07-31 17:55:31,849 - INFO - Model training complete.
2026-07-31 17:55:31,850 - INFO - Logging model parameters...
2026-07-31 17:55:32,290 - INFO - Making predictions...
2026-07-31 17:55:32,293 - INFO - Calculating evaluation metrics...
2026-07-31 17:55:32,324 - INFO - Logging evaluation metrics...
2026-07-31 17:55:34,134 - INFO - Saving and logging the model...
2026/07/31 17:55:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-31 17:56:01,972 - INFO - Model training and logging completed in 31.56 seconds.
2026-07-31 17:56:01,975 - INFO - Accuracy: 0.704
2026-07-31 17:56:01,979 - INFO - Precision: 0.7796610169491526
2026-07-31 17:56:01,982 - INFO - Recall: 0.6571428571428571
2026-07-31

🏃 View run burly-rat-770 at: https://dagshub.com/satyagudu1146/MLOPS-PROJ-CI-CD.mlflow/#/experiments/0/runs/525d25e1778e40368ca4d1a15e974b1d
🧪 View experiment at: https://dagshub.com/satyagudu1146/MLOPS-PROJ-CI-CD.mlflow/#/experiments/0
